# Embedding Visualization — UMAP + Plotly 3D Scatter

This notebook projects high-dimensional token/word embeddings into 3D using **UMAP** and visualizes them interactively with **Plotly**.

Concepts:
- Dimensionality reduction preserves local neighborhoods
- Color by semantic category to validate cluster structure
- Interactive zoom, rotate, and lasso selection

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.preprocessing import StandardScaler
import umap
import plotly.express as px

# Use a small synthetic embedding-like dataset for demo portability
np.random.seed(7)
n_per_class = 300
categories = ['electronics', 'biology', 'physics', 'literature']
high_dim = []
labels = []
for cat in categories:
    center = np.random.randn(64) * 2
    high_dim.append(np.random.randn(n_per_class, 64) + center)
    labels.extend([cat] * n_per_class)

X = np.vstack(high_dim)
X = StandardScaler().fit_transform(X)

# UMAP 3D projection
reducer = umap.UMAP(n_components=3, n_neighbors=15, min_dist=0.1, random_state=42)
embedding_3d = reducer.fit_transform(X)

df = pd.DataFrame({
    'UMAP-1': embedding_3d[:, 0],
    'UMAP-2': embedding_3d[:, 1],
    'UMAP-3': embedding_3d[:, 2],
    'Category': labels
})
df.head()

In [ ]:
# Interactive 3D scatter with Plotly
fig = px.scatter_3d(
    df, x='UMAP-1', y='UMAP-2', z='UMAP-3',
    color='Category',
    opacity=0.7,
    title='UMAP 3D Embedding Space',
    template='plotly_white',
    color_discrete_sequence=px.colors.qualitative.Vivid
)
fig.update_traces(marker=dict(size=3))
fig.update_layout(
    scene=dict(
        xaxis_title='UMAP-1',
        yaxis_title='UMAP-2',
        zaxis_title='UMAP-3'
    ),
    margin=dict(l=0, r=0, b=0, t=40)
)
fig.show()

## Interactive Nearest-Neighbor Probe

Click a point in the Plotly figure above, or use the slider below to pick a sample index and highlight its 5 nearest neighbors.

In [ ]:
from scipy.spatial.distance import cdist
from ipywidgets import interact, IntSlider
import plotly.graph_objects as go

def highlight_neighbors(idx=0, k=5):
    dists = cdist(embedding_3d[[idx]], embedding_3d)[0]
    neighbor_idx = np.argsort(dists)[1:k+1]
    df['NN'] = 'other'
    df.loc[neighbor_idx, 'NN'] = 'neighbor'
    df.loc[idx, 'NN'] = 'selected'
    
    fig2 = px.scatter_3d(
        df, x='UMAP-1', y='UMAP-2', z='UMAP-3',
        color='NN',
        color_discrete_map={'selected':'red', 'neighbor':'orange', 'other':'lightgray'},
        title=f'Sample {idx} + {k} nearest neighbors',
        template='plotly_white'
    )
    fig2.update_traces(marker=dict(size=[6 if n=='selected' else (5 if n=='neighbor' else 2) for n in df['NN']]))
    fig2.show()

interact(
    highlight_neighbors,
    idx=IntSlider(min=0, max=len(df)-1, step=1, value=0, description='Sample idx'),
    k=IntSlider(min=1, max=20, step=1, value=5, description='k neighbors')
)

## Expected Output

- A fully interactive 3D scatter plot with 4 clearly separated color clusters
- Lasso and box selection tools in the Plotly toolbar
- A slider-driven nearest-neighbor highlight that validates local structure preservation